# News Headline Generator (LSTM)

Trains a word-level Bidirectional LSTM on a small sample of real news headlines (taken from the public "A Million News Headlines" dataset) and then generates new headlines from a few starting words. The sample is included inline so it runs without Kaggle.

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `1 News Head line.ipynb`._

## 1. Imports

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

## 2. The data
One headline per line (the date column from the original CSV was removed).

In [ ]:
HEADLINES = """aba decides against community broadcasting licence
act fire witnesses must be aware of defamation
a g calls for infrastructure protection summit
air nz staff in aust strike for pay rise
air nz strike to affect australian travellers
ambitious olsson wins triple jump
antic delighted with record breaking barca
aussie qualifier stosur wastes four memphis match
aust addresses un security council over iraq
australia is locked into war timetable opp
australia to contribute 10 million in aid to iraq
barca take record as robson celebrates birthday in
bathhouse plans move ahead
big hopes for launceston cycling championship
big plan to boost paroo water supplies
blizzard buries united states in bills
brigadier dismisses reports troops harassed in
british combat troops arriving daily in kuwait
bryant leads lakers to double overtime win
bushfire victims urged to see centrelink
businesses should prepare for terrorist attacks
calleri avenges final defeat to eliminate massu
call for ethanol blend fuel to go ahead
carews freak goal leaves roma in ruins
cemeteries miss out on funds
code of conduct toughens organ donation regulations
commonwealth bank cuts fixed home loan rates
community urged to help homeless youth
council chief executive fails to secure position
councillor to contest wollongong as independent
council moves to protect tas heritage garden
council welcomes ambulance levy decision
council welcomes insurance breakthrough
crean tells alp leadership critics to shut up
dargo fire threat expected to rise
death toll continues to climb in south korean subway
dems hold plebiscite over iraqi conflict
dent downs philippoussis in tie break thriller
de villiers to learn fate on march 5
digital tv will become commonplace summit
direct anger at govt not soldiers crean urges
dispute over at smithton vegetable processing plant
dog mauls 18 month old toddler in nsw
dying korean subway passengers phoned for help
england change three for wales match
epa still trying to recover chemical clean up costs
expressions of interest sought to build livestock
fed opp to re introduce national insurance
firefighters contain acid spill
four injured in head on highway crash
freedom records net profit for third successive
funds allocated for domestic violence victims
funds allocated for youth at risk
funds announced for bridge work
funds to go to cadell upgrade
funds to help restore cossack
german court to give verdict on sept 11 accused
gilchrist backs rest policy
girl injured in head on highway crash
gold coast to hear about bilby project
golf club feeling smoking ban impact
govt is to blame for ethanols unpopularity opp
greens offer police station alternative
griffiths under fire over project knock back
group to meet in north west wa over rock art
hacker gains access to eight million credit cards
hanson is grossly naive over nsw issues costa
hanson should go back where she came from nsw mp
harrington raring to go after break
health minister backs organ and tissue storage
heavy metal deposits survey nearing end
injured rios pulls out of buenos aires open
inquest finds mans death accidental
investigations underway into death toll of korean
investigation underway into elster creek spill
iraqs neighbours plead for continued un inspections
iraq to pay for own rebuilding white house
irish man arrested over omagh bombing
irrigators vote over river management
israeli forces push into gaza strip
jury to consider verdict in murder case
juvenile sex offenders unlikely to reoffend as
kelly disgusted at alleged bp ethanol scare
kelly not surprised ethanol confidence low
korean subway fire 314 still missing
last minute call hands alinghi big lead
low demand forces air service cuts
man arrested after central qld hijack attempt
man charged over cooma murder
man fined after aboriginal tent embassy raid
man jailed over keno fraud
man with knife hijacks light plane
martin to lobby against losing nt seat in fed
massive drug crop discovered in western nsw
mayor warns landfill protesters
meeting to consider tick clearance costs
meeting to focus on broken hill water woes
moderate lift in wages growth
more than 40 pc of young men drink alcohol at
more water restrictions predicted for northern tas
more women urged to become councillors
most highly educated live in nsw wa
mp raises hospital concerns in parliament
mp rejects ambulance levy claims
mugabe to touch down in paris for summit
national gallery gets all clear after
nato gives green light to defend turkey
nca defends aboriginal tent embassy raid
new zealand imposes visa entry for zimbabwe
no side effects for new whooping cough vaccine
nsw govt under fire for holding back vegetation
nsw opp defends claims of running race campaign
nsw opp pledges 50m drought relief
nt govt boosts nurse number with overseas intake
nth koreans seek asylum at japanese embassy
nursing student intake down
oh brother your times up says ganguly senior
omodei to stay in politics
onesteel to invest 80m in whyalla steelworks
opposition urged to help protect recherche bay
orientation begins for uni students
osullivan in world cross country doubt
pagan says rule changes not necessary
pair to face court over ayr murder
patterson defends decision not to attend health
patterson no show displays govts arrogance crean
patterson snubs health meeting to avoid lions den
peace agreement may bring respite for venezuela
pienaar shines as ajax frustrate arsenal
plan for second skatepark
plan to encourage farmers into plantation timber
png nurses strike after colleague raped on way to
police cracking down on driver safety
police defend aboriginal tent embassy raid
policewomen accusations feature at federal crime
probe launched into plane crash
program to monitor forest harvested areas
public urged to check gas cylinders
public warned about phone scam
qantas international crews to strike over pay
qantas war plan to cut 2500 jobs outrages unions
qr not planning northern route sackings
questions public anger grows after korean subway
rabbit control program on trial
radioactive spill at wmcs olympic dam mine
rain eases wheatbelt water woes
reading go third in first division
record amount for gladstone ventures
refshauge wins defamation court case
regulator to inspect gm canola trials
report highlights container terminal potential
resource stocks boost all ords
restraint order issued against anti discrimination
rfs rejects claim that act authorities spurned
ricciuto undergoes surgery on injured ankle
rice mill closures to put 300 out of work
rsl angry about reports of troop harassment
safety review begins after bushwalker death
sa premier calls for action over river murray
saudi arabians to stand trial over al qaeda"""

lines = HEADLINES.strip().split("\n")
print(len(lines), "headlines")

## 3. Tokenize and create n-grams

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
def make_ngram_dataset(lines, word_to_id, max_len=None):
    """For each line 'a b c d' create the samples [a]->b, [a b]->c, [a b c]->d."""
    sequences = []
    for line in lines:
        ids = encode(line, word_to_id)
        for i in range(1, len(ids)):
            sequences.append(ids[: i + 1])
    if max_len is None:
        max_len = max(len(s) for s in sequences)
    padded = keras.utils.pad_sequences(sequences, maxlen=max_len, padding="pre", truncating="pre")
    X, y = padded[:, :-1], padded[:, -1]
    return X, y, max_len

In [ ]:
word_to_id, id_to_word = build_vocab(lines)
vocab_size = len(word_to_id) + 1
X, y, max_len = make_ngram_dataset(lines, word_to_id)
print("X:", X.shape, "| vocabulary:", vocab_size)

## 4. Model
The original notebook built the model but never compiled or trained it; here we do both.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(max_len - 1,)),
    layers.Embedding(vocab_size, 100),
    layers.Bidirectional(layers.LSTM(150, return_sequences=True)),
    layers.Dropout(0.2),
    layers.LSTM(150),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

## 5. Train

In [ ]:
history = model.fit(X, y, epochs=80, batch_size=32, verbose=2)

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
loss, acc = model.evaluate(X, y, verbose=0)
print(f"Training accuracy: {acc:.3f}")

## 7. Generate headlines
The original code looked up words with `list(word_index.keys())[index]`, which is off by one. Using an `id_to_word` dictionary fixes that.

In [ ]:
def predict_next_word(model, seed_text, word_to_id, id_to_word, max_len):
    ids = encode(seed_text, word_to_id)
    padded = keras.utils.pad_sequences([ids], maxlen=max_len - 1, padding="pre", truncating="pre")
    probs = model.predict(padded, verbose=0)[0]
    probs[0] = 0  # never predict the padding id
    return id_to_word.get(int(np.argmax(probs)), "")


def generate_text(model, seed_text, n_words, word_to_id, id_to_word, max_len):
    text = seed_text
    for _ in range(n_words):
        next_word = predict_next_word(model, text, word_to_id, id_to_word, max_len)
        if not next_word:
            break
        text += " " + next_word
    return text

In [ ]:
for seed in ["aussie qualifier stosur", "australia to", "police"]:
    print(generate_text(model, seed, 6, word_to_id, id_to_word, max_len))

In [ ]:
model.save("news_headline_lstm.keras")